In [1]:
# make sure jupyter server is installed in the environment
# then install dependencies
%pip install pandas nltk scikit-learn numpy matplotlib --quiet

from config import get_merged_dataframe
from main import configure

configure()

df = get_merged_dataframe(
    './data/processedNegative.csv',
    './data/processedPositive.csv',
    './data/processedNeutral.csv',
)


from tokenizer import (
    lemmatize_tokens,
    stem_tokens,
    snowball_stem_tokens,
    lancaster_stem_tokens,
    misspell_and_lemmatize_tokens,
    misspell_tokens
)
from cleaning import clean_tweets
from vectorizer import tfidf_vectorize, count_vectorize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB, ComplementNB
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

config = {
    "cleaning": {
        "Default Cleaning": clean_tweets,
    },
    "tokenization": {
        "Tokenization": None,
        "Lemmatization": lemmatize_tokens,
        "Stemming": stem_tokens,
        "Stemming Snowball": snowball_stem_tokens,
        "Stemming Lancaster": lancaster_stem_tokens,
        "Misspellings": misspell_tokens,
        "Misspellings + Lemmatization": misspell_and_lemmatize_tokens,
    },
    "vectorization": {
        "TF-IDF": tfidf_vectorize,
        "Count": count_vectorize,
    },
    "classifiers": [
        LogisticRegression(),
        RandomForestClassifier(),
        MultinomialNB(),
        SVC(),
        BernoulliNB(),
        ComplementNB(),
    ],
}

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt_tab to /home/samy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/samy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/samy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Similarity

In [2]:
for vectorizer_name, vectorizer_func in config["vectorization"].items():
    print(f"Vectorizer: {vectorizer_name}")
    for tokenizer_name, tokenizer_func in config["tokenization"].items():
        print(f"  Tokenizer: {tokenizer_name}")
        for cleaner_name, cleaner_func in config["cleaning"].items():
            print(f"    Cleaner: {cleaner_name}")
            print("10 similar tweets:")
            vectorizer, df_sample = vectorizer_func(
                df.copy(),
                text_column="tweet",
                cleaner=cleaner_func,
                tokenizer=tokenizer_func,
            )
            # apply cosine similarity and print 10 most similar tweets
            cosine_sim = cosine_similarity(df_sample)
            # get the upper triangle of the cosine similarity matrix - exclude self-similarity and diagonals
            # k=1 means "start 1 position above diagonal" (exclude diagonal)
            upper_tri_indices = np.triu_indices_from(cosine_sim, k=1)
            # extract similarity scores from upper triangle
            upper_tri_sim = cosine_sim[upper_tri_indices]
            
            # sort the similarity scores in descending order
            # -10 to get top 10 similar pairs in descending order
            # -1 means reverse order (descending)
            sorted_indices = np.argsort(upper_tri_sim)[-10:][::-1]
            for index in sorted_indices:
                i = upper_tri_indices[0][index]
                j = upper_tri_indices[1][index]
                sim_score = cosine_sim[i][j]
                print(f"      Tweet {i} {df['sentiment'].iloc[i]}: {df['tweet'].iloc[i]}")
                print(f"      Tweet {j} {df['sentiment'].iloc[j]}: {df['tweet'].iloc[j]}")
                print(f"      Similarity Score: {sim_score:.4f}")
                print()

/home/samy/tweets/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Vectorizer: TF-IDF
  Tokenizer: Tokenization
    Cleaner: Default Cleaning
10 similar tweets:
      Tweet 1832 positive: Thanks for the recent follow Happy to connect happy  have a great Thursday.  Want this
      Tweet 2455 positive: Thanks for the recent follow Happy to connect happy  have a great Thursday. Want this?
      Similarity Score: 1.0000

      Tweet 1868 positive: Thanks for the recent follow Happy to connect happy  have a great Thursday.  Want this ?
      Tweet 2455 positive: Thanks for the recent follow Happy to connect happy  have a great Thursday. Want this?
      Similarity Score: 1.0000

      Tweet 1832 positive: Thanks for the recent follow Happy to connect happy  have a great Thursday.  Want this
      Tweet 1868 positive: Thanks for the recent follow Happy to connect happy  have a great Thursday.  Want this ?
      Similarity Score: 1.0000

      Tweet 1745 positive: Thanks for the recent follow Happy to connect happy  have a great Thursday. Want this
      Twe